# Route B: BMZ BirdNET

CPU is enough. Runtime, Run all. About 5-10 minutes.

Uses the short real wav bundled with bacpipe (~1 min) unless you upload files to `/content/audio/`. Analysis bins are 15 s.

## 1. Clone

In [ ]:
REPO = "https://github.com/ST-48-1240162/bioacoustic-embedding-dynamics.git"

%cd /content
!rm -rf bioacoustic-embedding-dynamics
!git clone --depth 1 {REPO}
%cd bioacoustic-embedding-dynamics

## 2. Install

In [ ]:
import sys
!{sys.executable} -m pip install -q soundfile "bioacoustics-model-zoo[birdnet]"
!{sys.executable} -m pip install -q -r docs/colab-requirements.txt
!{sys.executable} -m pip install -q -e .

## 3. Audio

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "scripts")
from colab_wav_source import bacpipe_test_wav, install_bacpipe_for_test_wav

AUDIO_DIR = Path("/content/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
audio_files = sorted(AUDIO_DIR.glob("*.wav")) + sorted(AUDIO_DIR.glob("*.WAV"))
if not audio_files:
    install_bacpipe_for_test_wav()
    audio_files = [bacpipe_test_wav()]
print(len(audio_files), "file(s):", audio_files[0])

## 4. Manifest and analysis

In [ ]:
from pathlib import Path

from bioacoustic_embedding_dynamics.adapters import bmz_birdnet_to_manifest

MANIFEST = Path("data/bmz_birdnet.jsonl")
bmz_birdnet_to_manifest(audio_files, MANIFEST, batch_size=8, min_confidence=0.0)
print("lines:", sum(1 for _ in MANIFEST.open()))

In [ ]:
!python -m bioacoustic_embedding_dynamics.cli --manifest {MANIFEST} --out reports/bmz --seed 42 --bin-s 15

## 5. Summary and figures

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

summary = json.loads(Path("reports/bmz/summary.json").read_text())
print(json.dumps(summary, indent=2))

for name in [
    "pca_species.png", "umap_species.png", "trajectory_pca.png",
    "changepoints.png", "trajectory_changepoints.png", "hmm_regimes.png", "shuffle_null.png",
]:
    display(Image(filename=str(Path("reports/bmz") / name)))